# Train the watermark (objectives 1 and 3)

This notebook trains the StegaStamp-style encoder and decoder.

* **Objective 1.** The stamped 400×400 file decodes on its own.
* **Objective 3.** The same file still decodes after brightness, contrast, JPEG, and a small border crop (up to about 10% off each edge). The signal is pushed toward the center by the edge penalty, and training includes those crops.

The finder that locates a stamp inside a 1000×900 canvas is **not** trained here. Build that dataset later, from stamps this run produces.

Upload the whole `own` folder (it contains `stegwatermark/`) to Google Drive before running the real training cell. A T4 runtime is enough. Checkpoints go to Drive and include the optimizer and the step counter, so a disconnect can be resumed.

MIRFLICKR photos are mixed sizes. The loader fits each one to 400×400. They are not already 400×400.


## 1. Runtime and libraries

Use a GPU runtime. This does not reinstall PyTorch if Colab already has it.

In [4]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    %pip install -q torch torchvision
%pip install -q lpips opencv-python-headless pillow bchlib


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Drive and project folder

Mount Drive first. `PROJECT_ROOT` must be the directory that contains `stegwatermark/`.

In [1]:
# from pathlib import Path
# import sys

# DRIVE_OK = False
# try:
#     from google.colab import drive
#     drive.mount("/content/drive")
#     DRIVE_OK = True
# except Exception as error:
#     print("Drive not mounted (%s)." % error)

# CANDIDATES = [
#     Path("/content/drive/MyDrive/own"),
#     Path("/content/own"),
#     Path.cwd(),
#     Path.cwd().parent,
# ]
# PROJECT_ROOT = None
# for path in CANDIDATES:
#     if (path / "stegwatermark").is_dir():
#         PROJECT_ROOT = path
#         break
# if PROJECT_ROOT is None:
#     raise FileNotFoundError(
#         "Could not find stegwatermark/. Upload the own folder to Drive and set PROJECT_ROOT."
#     )
# sys.path.insert(0, str(PROJECT_ROOT))
# print("PROJECT_ROOT", PROJECT_ROOT)


from pathlib import Path
import sys

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = True
except Exception as error:
    print("Drive not mounted (%s)." % error)

CANDIDATES = [
    Path("/content/drive/MyDrive/GenAI-Image-Watermarking-main"),
    Path("/content/drive/MyDrive/own"),
    Path("/content/GenAI-Image-Watermarking-main"),
    Path.cwd(),
]
PROJECT_ROOT = None
for path in CANDIDATES:
    # Check directly inside the directory, or inside the nested folder structure
    if (path / "stegwatermark").is_dir():
        PROJECT_ROOT = path
        break
    elif (path / "GenAI-Image-Watermarking-main" / "stegwatermark").is_dir():
        PROJECT_ROOT = path / "GenAI-Image-Watermarking-main"
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find stegwatermark/ folder. Please check your path variables."
    )
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT successfully set to:", PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT successfully set to: /content/drive/MyDrive/own


## 3. Data and checkpoint directory

Mount Drive, then set `DATA_DIR` to the unzipped MIRFLICKR image folder. Copying it to `/content/data` makes the loader faster than reading every JPEG from Drive.

The real run is 140k steps, batch size 4. That is several hours on a T4. The smoke cell below only checks that one full forward/backward works.


In [9]:
DATA_DIR = "/content/data/mirflickr"          # folder of JPEG/PNG covers
SAVE_DIR = (
    "/content/drive/MyDrive/watermark_checkpoints"
    if DRIVE_OK
    else str(PROJECT_ROOT / "checkpoints")
)
print("DATA_DIR", DATA_DIR)
print("SAVE_DIR", SAVE_DIR)


DATA_DIR /content/data/mirflickr
SAVE_DIR /content/drive/MyDrive/watermark_checkpoints


## 4. Smoke test

Two steps on random images, with the image losses turned on, so the encoder, decoder, critic, warp, JPEG, and checkpoint all run once.

In [10]:
from stegwatermark.train_loop import TrainConfig, run_training

smoke = TrainConfig(
    synthetic=True,
    num_steps=2,
    batch_size=1,
    no_im_loss_steps=0,
    save_every=1,
    log_every=1,
    save_dir="/tmp/watermark_smoke",
    device="cuda",
)
run_training(smoke)
print("smoke test finished")


Using 64 synthetic images. Set data_dir to MIRFLICKR for a real run.
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
step 0  bit_acc 0.510  clean 0.740  secret 0.694  yuv 8.599  lpips 0.574  crop 0.000
saved /tmp/watermark_smoke/latest.pt
step 1  bit_acc 0.520  clean 0.620  secret 0.702  yuv 9.077  lpips 0.571  crop 0.000
saved /tmp/watermark_smoke/latest.pt
smoke test finished


I am trying to fix the error that is causing 9678 images to only load

In [2]:
import os
import random
import shutil
import zipfile

# 1. Source zip from your Drive screenshot
DRIVE_ZIP = "/content/drive/MyDrive/own/datasets/mirflickr25k.zip"
LOCAL_EXTRACT = "/content/mirflickr_raw"

# 2. Local output folders for fast training
LOCAL_TRAIN = "/content/data/train"
LOCAL_TEST = "/content/data/test"

if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f"Cannot find zip file at {DRIVE_ZIP}. Please double check path.")

# Extract zip to fast local disk
print("Extracting dataset from Google Drive to local runtime... (takes ~1 min)")
with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
    zf.extractall(LOCAL_EXTRACT)

# Gather and shuffle all extracted images
all_images = []
for root, dirs, files in os.walk(LOCAL_EXTRACT):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append(os.path.join(root, f))
all_images.sort()

print(f"Total images found in zip: {len(all_images)}")

# Perform the clean split on the fly
random.seed(42)
random.shuffle(all_images)

test_pool = all_images[:200]
train_pool = all_images[200:]

os.makedirs(LOCAL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_TEST, exist_ok=True)

for img in train_pool:
    shutil.move(img, os.path.join(LOCAL_TRAIN, os.path.basename(img)))
for img in test_pool:
    shutil.move(img, os.path.join(LOCAL_TEST, os.path.basename(img)))

print(f"✔ Success! Local Train: {len(os.listdir(LOCAL_TRAIN))} images")
print(f"✔ Success! Local Test: {len(os.listdir(LOCAL_TEST))} images")


Extracting dataset from Google Drive to local runtime... (takes ~1 min)
Total images found in zip: 25000
✔ Success! Local Train: 24800 images
✔ Success! Local Test: 200 images


In [12]:
import os
import zipfile

# Source location on Google Drive where your dataset zip file sits
DRIVE_ZIP = "/content/drive/MyDrive/own/datasets/mirflickr25k.zip"
LOCAL_TARGET = "/content/data"

if os.path.exists(DRIVE_ZIP):
    print("Extracting 2.86 GB dataset locally... (takes ~1-2 minutes)")
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        zf.extractall(LOCAL_TARGET)
    print("Extraction complete!")
else:
    raise FileNotFoundError(f"Could not locate zip file at: {DRIVE_ZIP}")


Extracting 2.86 GB dataset locally... (takes ~1-2 minutes)
Extraction complete!


## 5. Real training

`bit_acc` is measured after distortion and crop. `clean` is the same batch with no distortion. Clean accuracy should climb toward 0.95 first. Augmented accuracy follows as the ramps increase.

Loss weights match `scripts/base.sh`:

* steps 0–1500: message BCE only
* L2 (weight 2), LPIPS (weight 1.5), critic (weight 0.5) ramp over 15k steps
* Y/U/V scales 1 / 100 / 100
* warp ramps to ±10% of the width over 10k steps, empty borders filled white
* edge penalty starts at step 80k
* border-crop augmentation ramps to 10% after step 1500

To resume after a disconnect, set `RESUME` to `latest.pt`. The schedules continue from the saved step.


In [ ]:
config = TrainConfig(
    data_dir="/content/data/train",  # Points to the clean 24,800 images
    save_dir=SAVE_DIR,
    resume="",
    synthetic=False,
    batch_size=4,
    num_steps=140000,
    device="cuda",
)
run_training(config)


covers: 24800
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
step 0  bit_acc 0.468  clean 0.637  secret 0.732  yuv 0.000  lpips 0.000  crop 0.000
saved /content/drive/MyDrive/watermark_checkpoints/latest.pt
step 100  bit_acc 0.500  clean 0.510  secret 0.693  yuv 0.000  lpips 0.000  crop 0.000
step 200  bit_acc 0.500  clean 0.500  secret 0.693  yuv 0.000  lpips 0.000  crop 0.000
step 300  bit_acc 0.512  clean 0.517  secret 0.693  yuv 0.000  lpips 0.000  crop 0.000
step 400  bit_acc 0.510  clean 0.525  secret 0.693  yuv 0.000  lpips 0.000  crop 0.000
step 500  bit_acc 0.515  clean 0.520  secret 0.693  yuv 0.000  lpips 0.000  crop 0.000
step 600  bit_acc 0.472  clean 0.472  secret 0.693  yuv 0.000  lpips 0.000  crop 0.000
step 700  bit_acc 0.492  clean 0.492  secret 0.693  yuv 0.000  lpips 0.000  crop 0.000
step 800  bit_acc 0.520  clean 0.530  secret 0.693  yuv 0.000  lpips 0.

Below is the code for resuming work from 15000

In [ ]:
import os
from stegwatermark.train_loop import TrainConfig, run_training

# Target the checkpoint saved on your Drive at step 15,000
RESUME_PATH = "/content/drive/MyDrive/watermark_checkpoints/latest.pt"

# Dynamic save directory matching your initial configurations
SAVE_DIR = "/content/drive/MyDrive/watermark_checkpoints"

config = TrainConfig(
    data_dir="/content/data/train",
    save_dir=SAVE_DIR,
    resume=RESUME_PATH,              # Crucial: Loads your progress state!
    synthetic=False,
    batch_size=4,
    num_steps=140000,
    device="cuda",
)

print("Resuming loop structure from checkpoint...")
run_training(config)


Resuming loop structure from checkpoint...
covers: 24800
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 166MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
resumed at step 15001
step 15100  bit_acc 0.512  clean 0.512  secret 0.693  yuv 0.048  lpips 0.105  crop 0.068
step 15200  bit_acc 0.525  clean 0.525  secret 0.693  yuv 0.180  lpips 0.083  crop 0.069
step 15300  bit_acc 0.505  clean 0.505  secret 0.693  yuv 0.073  lpips 0.090  crop 0.069
step 15400  bit_acc 0.515  clean 0.515  secret 0.693  yuv 0.004  lpips 0.047  crop 0.069
step 15500  bit_acc 0.490  clean 0.490  secret 0.693  yuv 0.044  lpips 0.079  crop 0.070
step 15600  bit_acc 0.482  clean 0.482  secret 0.693  yuv 0.003  lpips 0.048  crop 0.070
step 15700  bit_acc 0.498  clean 0.498  secret 0.693  yuv 0.129  lpips 0.083  crop 0.071
step 15800  bit_acc 0.550  clean 0.550  secret 0.693  yuv 0.053  lpips 0.041  crop 0.071
step 15900  bit_acc 0.520  clean 0.520  secret 0.693  yuv 0.048  lpips 0.063  crop 0.072
step 16000  bit_acc 0.495  clean 0.495  secret 0.693  yuv 0.025  lpips 0.078  crop 0.072


## What this checkpoint can and cannot do

It can stamp a 400×400 image and read that image back, including after the light edits in objective 3.

It cannot find the stamp inside a 1000×900 screenshot. That is the finder, trained afterwards by pasting these stamps onto large canvases.
